In [ ]:
#@title Imports & Utils

import pandas as pd

import ast

import numpy as np

import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_validate, cross_val_predict, KFold, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_extraction.text import CountVectorizer

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import pandas as pd

import random

from IPython.display import clear_output

from collections import Counter

from sklearn.utils.class_weight import compute_class_weight

np.random.seed(123)
random.seed(123)

In [ ]:
#@title Dataset
distmult_test_preds = pd.read_csv("/content/distmult-test-preds.tsv", index_col=False, sep="\t").dropna().drop_duplicates().reset_index(drop=True)
distmult_val_preds = pd.read_csv("/content/distmult-val-preds.tsv", index_col=False, sep="\t").dropna().drop_duplicates().reset_index(drop=True)

transformer_test_preds = pd.read_csv("/content/roberta-test-preds.tsv", index_col=False, sep="\t").applymap(lambda x: str(x).strip()).dropna().drop_duplicates().reset_index(drop=True)
transformer_val_preds = pd.read_csv("/content/roberta-val-preds.tsv", index_col=False, sep="\t").applymap(lambda x: str(x).strip()).dropna().drop_duplicates().reset_index(drop=True)

assert len(distmult_test_preds) == len(transformer_test_preds)
assert len(distmult_val_preds) == len(transformer_val_preds)

# **Data Exploration**

In [ ]:
dictionary = {label: idx for idx, label in enumerate(set(distmult_test_preds['true'].values.tolist()))}
labels = list(dictionary.keys())
dictionary

In [ ]:
distmult_test_preds.head()

In [ ]:
transformer_test_preds.head()

In [ ]:
# Plotting Confusion Matrix for Transformer - test dataset
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(transformer_test_preds['true'].map(lambda x: dictionary.get(x)), \
                             transformer_test_preds['predicted'].map(lambda x: dictionary.get(x))), \
            annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels, fmt='g')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix Transformer - test set')

# Plotting Confusion Matrix for DistMult - test dataset
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(distmult_test_preds['true'].map(lambda x: dictionary.get(x)), \
                             distmult_test_preds['predicted'].map(lambda x: dictionary.get(x))), \
            annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels, fmt='g')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix DistMult - test set')

plt.tight_layout()
plt.show()

In [ ]:
# Plotting Confusion Matrix for Transformer - val dataset
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(transformer_val_preds['true'].map(lambda x: dictionary.get(x)), \
                             transformer_val_preds['predicted'].map(lambda x: dictionary.get(x))), \
            annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels, fmt='g')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix Transformer - val set')

# Plotting Confusion Matrix for DistMult - val dataset
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(distmult_val_preds['true'].map(lambda x: dictionary.get(x)), \
                             distmult_val_preds['predicted'].map(lambda x: dictionary.get(x))), \
            annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels, fmt='g')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix DistMult - val set')

plt.tight_layout()
plt.show()

In [ ]:
print("Transformer evaluation on test set: \n")
true = transformer_test_preds['true']
preds = transformer_test_preds['predicted']
print(f"F1-macro: {metrics.f1_score(true, preds, average='macro')}")
print(f"F1-micro: {metrics.f1_score(true, preds, average='micro')}")
print(f"Accuracy: {metrics.accuracy_score(true, preds)}")
print(f"Recall: {metrics.recall_score(true, preds, average='macro')}")
print(f"Precision: {metrics.precision_score(true, preds, average='macro')}")

In [ ]:
print("Transformer evaluation on val set: \n")
true = transformer_val_preds['true']
preds = transformer_val_preds['predicted']
print(f"F1-macro: {metrics.f1_score(true, preds, average='macro')}")
print(f"F1-micro: {metrics.f1_score(true, preds, average='micro')}")
print(f"Accuracy: {metrics.accuracy_score(true, preds)}")
print(f"Recall: {metrics.recall_score(true, preds, average='macro')}")
print(f"Precision: {metrics.precision_score(true, preds, average='macro')}")

In [ ]:
print("DistMult evaluation on test set: \n")
true = distmult_test_preds['true']
preds = distmult_test_preds['predicted']
print(f"F1-macro: {metrics.f1_score(true, preds, average='macro')}")
print(f"F1-micro: {metrics.f1_score(true, preds, average='micro')}")
print(f"Accuracy: {metrics.accuracy_score(true, preds)}")
print(f"Recall: {metrics.recall_score(true, preds, average='macro')}")
print(f"Precision: {metrics.precision_score(true, preds, average='macro')}")

In [ ]:
print("DistMult evaluation on val set: \n")
true = distmult_val_preds['true']
preds = distmult_val_preds['predicted']
print(f"F1-macro: {metrics.f1_score(true, preds, average='macro')}")
print(f"F1-micro: {metrics.f1_score(true, preds, average='micro')}")
print(f"Accuracy: {metrics.accuracy_score(true, preds)}")
print(f"Recall: {metrics.recall_score(true, preds, average='macro')}")
print(f"Precision: {metrics.precision_score(true, preds, average='macro')}")

# **In-Pipeline evaluation**
Simulating what will happen if models are stacked in a pipeline:



### 1. DistMult vs Transformer

Distmults predics which nodes don't have any relation and the Transformer (trained with multi-class classification) predics the rest

In [ ]:
no_rel_distmult = distmult_test_preds[distmult_test_preds['predicted'] == 'noRel']
len(no_rel_distmult)

In [ ]:
combined_dataset = pd.merge(distmult_test_preds, transformer_test_preds, on=['true', 'sentence1', 'sentence2'])
combined_dataset = combined_dataset.rename({'predicted_x': 'distmult_pred',\
                                            'predicted_y':'transformer_pred',\
                                            'true': 'y_true'}, axis=1)[['y_true', 'distmult_pred', 'transformer_pred']]
combined_dataset.head(2)

In [ ]:
combined_dataset['y_pred'] = np.where(combined_dataset['distmult_pred'] == 'noRel',
                                          combined_dataset['distmult_pred'],
                                          combined_dataset['transformer_pred'])
combined_dataset.head()

In [ ]:
print("DistMult f1-score")
print(metrics.f1_score(distmult_test_preds['true'], distmult_test_preds['predicted'], average='macro'))

print("Transformer f1-score")
print(metrics.f1_score(transformer_test_preds['true'], transformer_test_preds['predicted'], average='macro'))

print("Combined f1-score")
print(metrics.f1_score(combined_dataset['y_true'], combined_dataset['y_pred'], average='macro'))

### 2. Transformer vs DistMult

The transformer predics which nodes don't have any relation and DistMult (trained with multi-class classification) predics the rest

In [ ]:
combined_dataset = pd.merge(distmult_test_preds, transformer_test_preds, on=['true', 'sentence1', 'sentence2'])
combined_dataset = combined_dataset.rename({'predicted_x': 'distmult_pred',\
                                            'predicted_y':'transformer_pred',\
                                            'true': 'y_true'}, axis=1)[['y_true', 'distmult_pred', 'transformer_pred']]
combined_dataset['y_pred'] = np.where(combined_dataset['transformer_pred'] == 'noRel',
                                          combined_dataset['transformer_pred'],
                                          combined_dataset['distmult_pred'])
combined_dataset.head()

In [ ]:
print("DistMult f1-score")
print(metrics.f1_score(distmult_test_preds['true'], distmult_test_preds['predicted'], average='macro'))

print("Transformer f1-score")
print(metrics.f1_score(transformer_test_preds['true'], transformer_test_preds['predicted'], average='macro'))

print("Combined f1-score")
print(metrics.f1_score(combined_dataset['y_true'], combined_dataset['y_pred'], average='macro'))

# **Meta-Classifier**

In [ ]:
dictionary = {label: idx for idx, label in enumerate(set(distmult_test_preds['true'].values.tolist()))}
dictionary

In [ ]:
combined_transformer_preds = pd.concat([transformer_test_preds, transformer_val_preds], axis=0)
combined_distmult_preds = pd.concat([distmult_test_preds, distmult_val_preds], axis=0)

assert len(combined_distmult_preds) == len(combined_transformer_preds)

In [ ]:
combined_dataset_meta_learner = pd.merge(combined_distmult_preds, combined_transformer_preds, on=['true', 'sentence1', 'sentence2'])[['predicted_x', 'predicted_y', 'true']]
combined_dataset_meta_learner = combined_dataset_meta_learner.rename({'predicted_x': 'distmult_pred', 'predicted_y':'transformer_pred'}, axis=1)

combined_dataset_meta_learner = combined_dataset_meta_learner.applymap(lambda x: dictionary.get(x))

true = combined_dataset_meta_learner.pop('true')

In [ ]:
combined_dataset_meta_learner.dropna().drop_duplicates().reset_index(drop=True)
combined_dataset_meta_learner.head(5)

### Using only predictions from the two models

In [ ]:
X = combined_dataset_meta_learner
y = true

#### Decision Trees & Random Forests

In [ ]:
scoring = ['f1_macro', 'f1_micro', 'accuracy', 'precision_macro', 'recall_macro']

In [ ]:
clf = DecisionTreeClassifier(random_state=123)
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

In [ ]:
clf = RandomForestClassifier(max_depth=12, random_state=123)
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

In [ ]:
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
y_pred = cross_val_predict(clf, X, y, cv=cv)
best_results = pd.DataFrame({'distmult_pred': X['distmult_pred'],\
                             'transformer_pred': X['transformer_pred'],\
                             'meta_pred': y_pred ,\
                             'true': y})
best_results.to_csv('best_results_meta_learner.csv', index=False)

#### Support Vector Classifiers

In [ ]:
clf = SVC(kernel='rbf', gamma='scale', random_state=123, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### Ada Boost Classifier

In [ ]:
clf = AdaBoostClassifier(n_estimators=100, random_state=123)
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### Gaussian Naive Bayes

In [ ]:
clf = GaussianNB()
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### K-Neighbors Classifier

In [ ]:
clf = KNeighborsClassifier(n_neighbors=5)
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### MLP

In [ ]:
clf = MLPClassifier(random_state=123, max_iter=300)
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### Gradient Boosting Machines

In [ ]:
clf = GradientBoostingClassifier()
cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
scores = cross_validate(clf, X, y, scoring=scoring, cv=cv)

print(f"F1-macro: {scores['test_f1_macro'].mean()}")
print(f"F1-micro: {scores['test_f1_micro'].mean()}")
print(f"Accuracy: {scores['test_accuracy'].mean()}")
print(f"Precision-macro: {scores['test_precision_macro'].mean()}")
print(f"Recall-macro: {scores['test_recall_macro'].mean()}")

#### Hypertuning

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(combined_dataset_meta_learner, true, test_size=0.2, random_state=123)

##### Random Forests

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'criterion': ['gini', 'entropy'],  # Function to measure the quality of a split
    'max_depth': [None, 5, 10, 20],  # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'random_state': [123],  # Random seed for reproducibility
}

rf_classifier = RandomForestClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
grid_search = GridSearchCV(estimator=rf_classifier, param_grid=param_grid, cv=cv, scoring='f1_macro')

grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found:", best_params)

# Predict on test data using the best estimator
y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Test F1-macro of the best model: {metrics.f1_score(y_test, y_pred, average='macro')}")

##### Gaussian Naive Bayes

In [ ]:
param_grid = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5],  # Portion of the largest variance of all features that is added to variances for calculation stability
}

cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
grid_search = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=cv, scoring='f1_macro')
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found:", best_params)

# Predict on test data using the best estimator
y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Test F1-macro of the best model: {metrics.f1_score(y_test, y_pred, average='macro')}")

##### Gradient Boosting Classifier

In [ ]:
param_grid = {
    'learning_rate': [0.01, 0.1, 0.5],  # Step size shrinkage used in update to prevent overfitting
    'n_estimators': [50, 100, 200],  # Number of boosting stages to be run
    'max_depth': [3, 5, 7],  # Maximum depth of the individual estimators
    'min_samples_split': [2, 5, 10],  # The minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # The minimum number of samples required to be at a leaf node
    'random_state': [123],  # Random seed for reproducibility
}

cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
grid_search = GridSearchCV(estimator=GradientBoostingClassifier(), param_grid=param_grid, cv=cv, scoring='f1_macro')
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found:", best_params)

# Predict on test data using the best estimator
y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Test F1-macro of the best model: {metrics.f1_score(y_test, y_pred, average='macro')}")

##### Ada Boost

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300],  # Number of boosting stages to be run
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5],  # Weight learning rate (shrinks the contribution of each classifier)
    'algorithm': ['SAMME', 'SAMME.R'],  # Algorithm used for weight updating
    'random_state': [123],  # Random seed for reproducibility
}

cv = StratifiedKFold(n_splits=5, random_state=123, shuffle=True)
grid_search = GridSearchCV(estimator=AdaBoostClassifier(), param_grid=param_grid, cv=cv, scoring='f1_macro')
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found:", best_params)

# Predict on test data using the best estimator
y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Test F1-macro of the best model: {metrics.f1_score(y_test, y_pred, average='macro')}")

### Using also the text

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')

print(device)
dictionary

In [ ]:
class CNN_Text(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super(CNN_Text, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv1 = nn.Conv2d(1, 100, (3, embedding_dim))
        self.fc1 = nn.Linear(100 + 2, 50)  # 100 for text output + 2 for numerical features
        self.fc2 = nn.Linear(50, num_classes)

    def forward(self, x_text, x_num1, x_num2):
        x_text = self.embedding(x_text)
        x_text = x_text.unsqueeze(1)
        x_text = F.relu(self.conv1(x_text)).squeeze(3)
        x_text = F.max_pool1d(x_text, x_text.size(2)).squeeze(2)

        # Concatenate text output with numerical features
        x = torch.cat((x_text, x_num1.unsqueeze(1), x_num2.unsqueeze(1)), dim=1)
        x = self.fc1(x)
        x = self.fc2(F.relu(x))
        return x

In [ ]:
class CNN_Dataset(Dataset):
    def __init__(self, X_num1, X_num2, X_text, y):
        self.X_num1 = torch.tensor(X_num1, dtype=torch.float32)
        self.X_num2 = torch.tensor(X_num2, dtype=torch.float32)
        self.X_text = torch.tensor(X_text, dtype=torch.long)

        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num1[idx].to(device),\
                self.X_num2[idx].to(device),\
                self.X_text[idx].to(device),\
                self.y[idx].to(device)

In [ ]:
# Preparing dataset
combined_dataset = pd.DataFrame({'y_true': combined_distmult_preds['true'].map(lambda x: dictionary.get(x)),
                                 'distmult_pred': combined_distmult_preds['predicted'].map(lambda x: dictionary.get(x)),
                                 'transformer_pred': combined_transformer_preds['predicted'].map(lambda x: dictionary.get(x)),
                                 'text': combined_distmult_preds['sentence1'] + ' - ' + combined_distmult_preds['sentence2']}).sample(frac=1)

X_num1 = combined_dataset['distmult_pred'].values
X_num2 = combined_dataset['transformer_pred'].values
y = combined_dataset['y_true'].values

vectorizer = CountVectorizer()
corpus = combined_dataset['text'].tolist()
X_text = vectorizer.fit_transform(corpus).astype(np.float32).toarray()


print(len(X_num1))
print(len(X_num2))
print(len(X_text))
X_num1

In [ ]:
# Define hyperparameters
vocab_size = len(vectorizer.vocabulary_)
embedding_dim = 100
num_classes = 3
lr = 0.001
batch_size = 32
num_epochs = 30

# Initialize lists to store predictions from each fold
all_y_true = []
all_y_pred = []
all_distmult_pred = []
all_transformer_pred = []

In [ ]:
num_folds = 5
cv = StratifiedKFold(n_splits=num_folds, random_state=123, shuffle=True)

for fold, (train_index, test_index) in enumerate(cv.split(X_num1, y)):
    print(f"Fold {fold + 1}/{num_folds}")

    train_dataset = CNN_Dataset(X_num1[train_index], X_num2[train_index], X_text[train_index], y[train_index])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    test_dataset = CNN_Dataset(X_num1[test_index], X_num2[test_index], X_text[test_index], y[test_index])
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = CNN_Text(vocab_size, embedding_dim, num_classes)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_loss = float("inf")
    best_model_path = f'best_model_fold_{fold}.pth'

    for epoch in range(num_epochs):
        model.train()
        loop = tqdm(train_loader, leave=True)
        avg_loss = 0
        for batch in loop:
            optimizer.zero_grad()
            X_num1_batch, X_num2_batch, X_text_batch, y_batch = batch
            outputs = model(X_text_batch, X_num1_batch, X_num2_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            loop.set_description(f"Epoch {epoch+1}/{num_epochs}, Fold {fold + 1}/{num_folds}")
            loop.set_postfix(loss=loss.item())
            avg_loss += loss.item()

        avg_loss = avg_loss/len(train_loader)
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), best_model_path)

        print(f"Epoch {epoch+1}/{num_epochs}, Fold {fold + 1}/{num_folds}, Loss: {avg_loss:.4f}")


    # Load the best model for prediction for this fold
    best_model = CNN_Text(vocab_size, embedding_dim, num_classes)
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.to(device)
    best_model.eval()

    fold_y_true = []
    fold_y_pred = []
    fold_distmult_pred = []
    fold_transformer_pred = []

    with torch.no_grad():
        for batch in test_loader:
            X_num1_batch, X_num2_batch, X_text_batch, y_batch = batch
            outputs = best_model(X_text_batch, X_num1_batch, X_num2_batch)
            _, predicted = torch.max(outputs.data, 1)
            fold_y_true.extend(y_batch.cpu().numpy())
            fold_y_pred.extend(predicted.cpu().numpy())
            fold_distmult_pred.extend(X_num1_batch.cpu().numpy())
            fold_transformer_pred.extend(X_num2_batch.cpu().numpy())

    all_y_true.append(fold_y_true)
    all_y_pred.append(fold_y_pred)
    all_distmult_pred.append(fold_distmult_pred)
    all_transformer_pred.append(fold_transformer_pred)

In [ ]:
f1_score = []
precision = []
recall = []
for y_true, y_pred in zip(all_y_true, all_y_pred):
  f1_score.append(metrics.f1_score(y_true, y_pred, average='macro'))
  precision.append(metrics.precision_score(y_true, y_pred, average='macro', zero_division=0))
  recall.append(metrics.recall_score(y_true, y_pred, average='macro', zero_division=0))

print(f1_score)
print(precision)
print(recall)

print(f"Macro F1 Score: {np.mean(f1_score):.4f}")
print(f"Macro Precision: {np.mean(precision):.4f}")
print(f"Macro Recall: {np.mean(recall):.4f}")

In [ ]:
best_results = pd.DataFrame({'distmult_pred': [x for xs in all_distmult_pred for x in xs],\
                             'transformer_pred': [x for xs in all_transformer_pred for x in xs],\
                             'meta_pred': [x for xs in all_y_pred for x in xs] ,\
                             'true': [x for xs in all_y_true for x in xs]})
best_results.to_csv('best_results_meta_learner.csv', index=False)

# Averaging Models

In [ ]:
transformer_f1 = metrics.f1_score(transformer_test_preds['true'], transformer_test_preds['predicted'], average='macro')
distmult_f1 = metrics.f1_score(distmult_test_preds['true'], distmult_test_preds['predicted'], average='macro')

print(transformer_f1, distmult_f1)

In [ ]:
weight_transformer = transformer_f1 / (transformer_f1 + distmult_f1)
weight_distmult = distmult_f1 / (transformer_f1 + distmult_f1)

print(weight_transformer, weight_distmult)

In [ ]:
combined_data = pd.DataFrame({
    'true' : transformer_test_preds['true'],
    'transformer_support_proba': transformer_test_preds['Support_proba'].map(lambda x: float(x) * weight_transformer),
    'distmult_support_proba': distmult_test_preds['Support_proba'].map(lambda x: float(x) * weight_distmult),
    'transformer_attack_proba': transformer_test_preds['Attack_proba'].map(lambda x: float(x) * weight_transformer),
    'distmult_attack_proba': distmult_test_preds['Attack_proba'].map(lambda x: float(x) * weight_distmult),
    'transformer_noRel_proba': transformer_test_preds['noRel_proba'].map(lambda x: float(x) * weight_transformer),
    'distmult_noRel_proba': distmult_test_preds['noRel_proba'].map(lambda x: float(x) * weight_distmult)
})

In [ ]:
combined_data['Support_proba'] =  combined_data['transformer_support_proba'] + combined_data['distmult_support_proba']
combined_data['Attack_proba'] =  combined_data['transformer_attack_proba'] + combined_data['distmult_attack_proba']
combined_data['noRel_proba'] =  combined_data['transformer_noRel_proba'] + combined_data['distmult_noRel_proba']

In [ ]:
probabilities = combined_data[['Support_proba', 'Attack_proba', 'noRel_proba']].values.tolist()
dictionary = {0: "Support", 1: "Attack", 2: "noRel"}

In [ ]:
combined_data['predicted'] = [dictionary.get(np.argmax(row)) for row in probabilities]

In [ ]:
# Calculate F1 score, precision, and recall
f1 = metrics.f1_score(combined_data['true'], combined_data['predicted'], average='macro')
precision = metrics.precision_score(combined_data['true'], combined_data['predicted'], average='macro', zero_division=0)
recall = metrics.recall_score(combined_data['true'], combined_data['predicted'], average='macro', zero_division=0)

print(f"Macro F1 Score: {f1:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")

# Accuracy analysis

In [ ]:
dictionary
dictionary_reverse = {v: k for k, v in dictionary.items()} #inverted
dictionary_reverse

In [ ]:
best_results.replace(dictionary_reverse, inplace=True)

In [ ]:
count_same_values = (best_results['distmult_pred'] == best_results['transformer_pred']).sum()
print(f"Distmult and transformer agree: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] == best_results['transformer_pred']) &
                     (best_results['transformer_pred'] == best_results['meta_pred'])).sum()
print(f"All models agree: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] == best_results['transformer_pred']) & \
                     (best_results['transformer_pred'] == best_results['true'])).sum()
print(f"Distmult and transformer agree and result is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] == best_results['transformer_pred']) & \
                     (best_results['distmult_pred'] == best_results['true']) & \
                     (best_results['transformer_pred'] == best_results['meta_pred'])).sum()
print(f"All models agree and result is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['transformer_pred'] != best_results['meta_pred'])).sum()
print(f"Meta disagree from transformer: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] != best_results['meta_pred'])).sum()
print(f"Meta disagree from distmult: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] != best_results['meta_pred']) & \
                     (best_results['transformer_pred'] != best_results['meta_pred'])).sum()
print(f"Meta disagree from distmult and transformer: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['transformer_pred'] != best_results['meta_pred']) & \
                     (best_results['meta_pred'] == best_results['true'])).sum()
print(f"Meta disagree from transformer and result is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] != best_results['meta_pred']) & \
                     (best_results['meta_pred'] == best_results['true'])).sum()
print(f"Meta disagree from distmult and result is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['meta_pred'] == best_results['true'])).sum()
print(f"Meta is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['distmult_pred'] == best_results['true'])).sum()
print(f"DistMult is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
count_same_values = ((best_results['transformer_pred'] == best_results['true'])).sum()
print(f"Transformer is true: {round(count_same_values/len(best_results)*100, 3)}%")

In [ ]:
best_results['distmult_pred'].value_counts() / len(best_results)

In [ ]:
best_results['transformer_pred'].value_counts() / len(best_results)

In [ ]:
best_results['meta_pred'].value_counts() / len(best_results)

In [ ]:
best_results['true'].value_counts() / len(best_results)

In [ ]:
labels = list(dictionary.keys())
print(labels)

meta_name = "RFC"
tranformer_name = "RoBERTa-V3"

cms = [
    confusion_matrix(best_results['true'], best_results['distmult_pred']),
    confusion_matrix(best_results['true'], best_results['transformer_pred']),
    confusion_matrix(best_results['true'], best_results['meta_pred'])
]
titles = ['Confusion Matrix DistMult',
          f'Confusion Matrix {tranformer_name}',
          f'Confusion Matrix \n {meta_name} Meta-Learner']

# Plotting
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (cm, title) in enumerate(zip(cms, titles)):
    sns.heatmap(cm, annot=True, cmap=sns.cm.rocket_r, fmt='g', ax=axes[i], xticklabels=labels, yticklabels=labels)
    axes[i].set_xlabel('Predicted labels')
    axes[i].set_ylabel('True labels')
    axes[i].set_title(title, weight='bold', fontdict={'fontsize':18})

plt.tight_layout()
plt.savefig(f'confusion-matrix-{tranformer_name}.pdf', dpi=1200)